In [74]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
pd.set_option('display.max_rows', 20)

In [75]:
# 1. Создаём свой класс Dataset для PyTorch
class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Токенизируем текст
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [76]:
import pandas as pd

df = pd.read_csv('../dataset/extended_by_deepseek_3.csv', sep=',')
df = df[df['тема'].notna()]
df = df[df['текст'].notna()]
df = df[['тема', 'текст']]

# темы, по которым пока не был получен ответ от deepseek:
df = df[df['тема'] != "вышка тура"]
df = df[df['тема'] != "плиткорез электрический"]
df = df[df['тема'] != "реноватор аккумуляторный"]
df = df[df['тема'] != "присоска"]
df = df[df['тема'] != "нивелир оптический"]
df = df[df['тема'] != "детектор проводки"]
df = df[df['тема'] != "дрель магнитная"]

# df = df[df['тема'] != "не вопрос клиента"]
df

,тема,текст
0,дрель-шуруповерт,"добрый вечер! подскажите, у вас можно взять в ..."
1,пылесос,Здравствуйте) сколько стоит аренда строительно...
2,плиткорез ручной,Доброе утро. Подскажите пожалуйста в какую цен...
3,фен,Доброе утро Есть строительный фен на аккумулят...
4,не вопрос клиента,Добрый день
...,...,...
1492,пароочиститель,"'Здравствуйте, есть ли в прокате устройство д..."
1493,пароочиститель,"'Привет, сколько стоит арендовать пароочистит..."
1494,пароочиститель,"'Добрый день, нужен пароочиститель для уборки..."
1495,пароочиститель,"'Здравствуйте, подскажите, можно ли взять пар..."


In [77]:
def clear_text(text: str) -> str:
    # функция очистки текстов от паразитных слов, не относящихся к теме запроса
    
    trash_values = [
        'добрый вечер', 
        'добрый день', 
        'доброе утро',
        'добрый утро',
        'здравствуйте',
        'подскажите',
        'пожалуйста',
        'хотелось',
        'интересует',
        'нужен',
        'нужно',
        'хочу',
        'надо',
        'можно',
        'аренда',
        'аренду',
        'хочу',
        'хотел бы',
        'в наличии',
        'сколько стоит',
        ',',
        "'",
    ]
    text = text.lower()
    for value in trash_values:
        text = text.replace(value, '')
    return text

In [78]:
df['текст'] = df['текст'].map(clear_text)
df

,тема,текст
0,дрель-шуруповерт,! у вас взять в шурповерт?
1,пылесос,) строительного пылесоса на сутки?
2,плиткорез ручной,. в какую цену у вас следующие инструменты: ...
3,фен,есть строительный фен на аккумуляторной батареи?
4,не вопрос клиента,
...,...,...
1492,пароочиститель,есть ли в прокате устройство для чистки паро...
1493,пароочиститель,привет арендовать пароочиститель для чистки ...
1494,пароочиститель,пароочиститель для уборки после ремонта есть ?
1495,пароочиститель,ли взять пароочиститель на сутки?


In [79]:
df['номер темы'] = df['тема'].astype('category').cat.codes
df

,тема,текст,номер темы
0,дрель-шуруповерт,! у вас взять в шурповерт?,19
1,пылесос,) строительного пылесоса на сутки?,59
2,плиткорез ручной,. в какую цену у вас следующие инструменты: ...,51
3,фен,есть строительный фен на аккумуляторной батареи?,76
4,не вопрос клиента,,36
...,...,...,...
1492,пароочиститель,есть ли в прокате устройство для чистки паро...,42
1493,пароочиститель,привет арендовать пароочиститель для чистки ...,42
1494,пароочиститель,пароочиститель для уборки после ремонта есть ?,42
1495,пароочиститель,ли взять пароочиститель на сутки?,42


In [80]:
df_small = df[df['тема']=='бензорез']
df_small

,тема,текст,номер темы
34,бензорез,. бензорез у вас есть?,5
254,бензорез,у вас есть в бетонорез?,5
678,бензорез,бензорез под 400 диск хускварна. есть такой?,5
809,бензорез,. с большим диском по бетону на сутки по како...,5
881,бензорез,есть ли бензорез с 400 диском в на одни сутки?,5
1076,бензорез,бензорез для резки металла на завтра ?,5
1077,бензорез,\n есть ли у вас бензорез для демонтажа заб...,5
1078,бензорез,\nпривет взять бензорез на пару часов чтобы...,5
1079,бензорез,\n ли арендовать бензорез на выходные?,5
1080,бензорез,\n взять бензорез на сутки резать арматуру?,5


In [81]:
summary = df['тема'].value_counts()
print(summary)

тема
не вопрос клиента                126
перфоратор                        78
пылесос                           65
дрель-шуруповерт                  55
торцовочная пила                  45
                                ... 
сабельная пила аккумуляторная     11
домкрат подкатной                 11
миксер                            10
сварочный аппарат                 10
Отбойный молоток                  10
Name: count, Length: 82, dtype: int64


In [82]:
# Применяем маску: оставляем только те строки, где размер темы >= 5
mask = df.groupby('тема')['тема'].transform('size') >= 5
df = df[mask]
df['номер темы'] = df['тема'].astype('category').cat.codes
df

,тема,текст,номер темы
0,дрель-шуруповерт,! у вас взять в шурповерт?,19
1,пылесос,) строительного пылесоса на сутки?,59
2,плиткорез ручной,. в какую цену у вас следующие инструменты: ...,51
3,фен,есть строительный фен на аккумуляторной батареи?,76
4,не вопрос клиента,,36
...,...,...,...
1492,пароочиститель,есть ли в прокате устройство для чистки паро...,42
1493,пароочиститель,привет арендовать пароочиститель для чистки ...,42
1494,пароочиститель,пароочиститель для уборки после ремонта есть ?,42
1495,пароочиститель,ли взять пароочиститель на сутки?,42


In [83]:
summary = df['тема'].value_counts()
print(summary)

тема
не вопрос клиента                126
перфоратор                        78
пылесос                           65
дрель-шуруповерт                  55
торцовочная пила                  45
                                ... 
сабельная пила аккумуляторная     11
домкрат подкатной                 11
миксер                            10
сварочный аппарат                 10
Отбойный молоток                  10
Name: count, Length: 82, dtype: int64


In [84]:
labels_df = df[['номер темы', 'тема']]
labels_df = labels_df.drop_duplicates(subset=['номер темы'])
labels_df = labels_df.sort_values(by='номер темы')
labels_df

,номер темы,тема
906,0,Отбойный молоток
39,1,УШМ
41,2,УШМ аккумуляторная
361,3,бензобур
330,4,бензопила
...,...,...
105,77,фен аккумуляторный
155,78,фрезер
163,79,шлифовальная машина
636,80,шнек для бензобура


In [85]:
labels_df.to_csv('../dataset/labels.csv')

In [86]:
# 2. Загружаем модель и токенизатор
import os
with open('hf_token.txt', encoding='utf-8') as f:
    token = f.read()
os.environ["HF_TOKEN"] = token
# model_name = "cointegrated/rubert-tiny2"
model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(labels_df))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [87]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(df, test_size=0.1, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 1346, Val: 75, Test: 75


In [88]:
train_texts = train['текст'].tolist()
train_labels = train['номер темы'].tolist()
val_texts = val['текст'].tolist()
val_labels = val['номер темы'].tolist()
test_texts = test['текст'].tolist()
test_labels = test['номер темы'].tolist()

train_dataset = IntentDataset(train_texts, train_labels, tokenizer)
val_dataset = IntentDataset(val_texts, val_labels, tokenizer)

In [89]:
# 4. Настраиваем параметры обучения
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=20,          
    per_device_train_batch_size=4, # маленький батч для CPU
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    dataloader_pin_memory=False,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

In [90]:
# 5. Функция для подсчёта метрик
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {
        'accuracy': acc,
        'f1': f1
    }

In [91]:
# 6. Запускаем обучение
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("🚀 Начинаем обучение...")
trainer.train()

🚀 Начинаем обучение...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,3.638100,3.658998,0.173333,0.107190
2,2.656200,2.577691,0.480000,0.416931
3,1.878800,1.545756,0.666667,0.618878
4,0.600000,0.969533,0.800000,0.799259
5,0.713900,0.758397,0.840000,0.850370
6,0.240400,0.795887,0.840000,0.843704
7,0.234400,0.786646,0.840000,0.834532
8,0.113700,0.688117,0.866667,0.882751
9,0.244800,0.605746,0.893333,0.897481
10,0.013400,0.551554,0.920000,0.920593


TrainOutput(global_step=4381, training_loss=0.9131844186491259, metrics={'train_runtime': 2693.8167, 'train_samples_per_second': 9.993, 'train_steps_per_second': 2.502, 'total_flos': 1151806045977600.0, 'train_loss': 0.9131844186491259, 'epoch': 13.0})

In [92]:
# 7. Сохраняем модель
dir_name = model_name.replace('/', '_')
model.save_pretrained(os.path.join('..', 'models', dir_name))
tokenizer.save_pretrained(os.path.join('..', 'models', dir_name))

print("✅ Модель сохранена")

✅ Модель сохранена


In [94]:
device = torch.device("cpu")

# 8. Тестируем модель на новых фразах
def predict_intent(text, model, tokenizer):
    model.to(device)
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.softmax(outputs.logits, dim=-1)
        # print(predictions)
        predicted_class = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_class].item()
    
    return predicted_class, confidence

acc = 0
for test_text, test_label in zip(test_texts, test_labels):
    # print(test_text)
    pred, conf = predict_intent(test_text, model, tokenizer)

    if pred==test_label:
        symbol = '🍏'
    else:
        symbol = '🍎'
    
    print(f"{symbol} '{test_text}' -> {labels_df.loc[labels_df['номер темы'] == pred, 'тема'].values[0]} (уверенность: {conf:.2f})")
    acc += pred==test_label
    print (labels_df.loc[labels_df['номер темы'] == test_label, 'тема'])
print(f'Точность: {acc/len(test_texts)}')

🍏 '  полировальной машинки ?' -> машина полировальная (уверенность: 0.99)
12    машина полировальная
Name: тема, dtype: object
🍏 '!   распиловочный станок?' -> станок распиловочный (уверенность: 0.98)
417    станок распиловочный
Name: тема, dtype: object
🍏 '!  у вас взять болгарку (ушм) на два дня? сколько будет стоить? и  ли залог?' -> УШМ (уверенность: 0.99)
39    УШМ
Name: тема, dtype: object
🍏 '  нужна тачка для работы на стройке  ' -> тачка (уверенность: 0.99)
279    тачка
Name: тема, dtype: object
🍏 '!  на завтра забронировать строительный миксер ручной? скорее всего на сутки т.к. до 18-00 не успею вернуть.' -> миксер (уверенность: 0.99)
66    миксер
Name: тема, dtype: object
🍏 ' есть ли бензорез с 400 диском в  на одни сутки?' -> бензорез (уверенность: 0.99)
34    бензорез
Name: тема, dtype: object
🍎 '.  с большим диском по бетону на сутки по какой цене будет?' -> гвоздезабивной пистолет по бетону (уверенность: 0.72)
34    бензорез
Name: тема, dtype: object
🍏 '    краскопульта  